# CKA Circuit Comparison (IID vs Non-IID)

Use **Centered Kernel Alignment (CKA)** to compare how models and circuits trained under different partitions (e.g., IID vs Non-IID) encode classes.

**Note:** This notebook does NOT train models from scratch. It assumes you have already trained your models and discovered circuits using `FedMI`. You will upload your experiment artifacts (`config.json`, `checkpoints`, `circuits`) to run the comparison.

---

## 1 · Setup environment

In [ ]:
import os
REPO_URL = "https://github.com/ha405/FedMI.git"
BRANCH = "cvpr"

if not os.path.isdir("FedMI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd FedMI && git pull origin {BRANCH}

os.chdir("FedMI")

from fedmi.env import setup, patch_config, print_info
setup()
print_info()

## 2 · Instructions to Upload Your Experiments

You need two pre-computed experiment directories to compare. For this example, let's call them `exp_iid` and `exp_noniid`.

For each experiment folder, ensure it contains the following structure:
- `config.json`
- `checkpoints/` -> (containing `checkpoint_round_X.pt`)
- `circuits/` -> (containing `all_circuits.json`)

**Run the cell below** to create the folders. Then, manually drag and drop your files into these folders using the left sidebar.

In [ ]:
exp_a_path = "exp_iid"
exp_b_path = "exp_noniid"

os.makedirs(exp_a_path, exist_ok=True)
os.makedirs(exp_b_path, exist_ok=True)

print(f"Folders created: '{exp_a_path}' and '{exp_b_path}'.")
print("Please upload your 'config.json', 'checkpoints/', and 'circuits/' to these folders before proceeding. You can just zip them and upload them and unzip them using !unzip commands if desired.")

## 3 · Verify Uploaded Files
Before running CKA, let's verify you placed the files correctly.

In [ ]:
def verify_exp(exp_path):
    if not os.path.exists(os.path.join(exp_path, "config.json")):
        print(f"[Error] Missing config.json in {exp_path}")
    if not os.path.exists(os.path.join(exp_path, "checkpoints")):
        print(f"[Error] Missing checkpoints folder in {exp_path}")
    if not os.path.exists(os.path.join(exp_path, "circuits", "all_circuits.json")):
        print(f"[Error] Missing circuits/all_circuits.json in {exp_path}")
    print(f"Verified {exp_path} structure looks okay!")
    
verify_exp(exp_a_path)
verify_exp(exp_b_path)

## 4 · CKA Comparison: Pre-head Latents

Compare the full-model representations (before the classification head) between the two experiments. Do they encode the inputs securely or in visually different latent spaces?

In [ ]:
from fedmi.playground import CKACompareExperiment

class Args:
    pass

args_prehead = Args()
args_prehead.exp_a = exp_a_path
args_prehead.exp_b = exp_b_path
args_prehead.client_a = 0
args_prehead.client_b = 0
args_prehead.mode = "prehead"
args_prehead.round = None
args_prehead.max_samples = 2000
args_prehead.output = None  # Heatmap skipped for single scalar

cka_exp_prehead = CKACompareExperiment(args_prehead)
cka_exp_prehead.run()

## 5 · CKA Comparison: Cross-Experiment Circuit Matching

Compare the activated circuit for matching classes across the two experiments. A low score implies the networks built structurally or functionally isolated pathways for the same class.

In [ ]:
args_circuit = Args()
args_circuit.exp_a = exp_a_path
args_circuit.exp_b = exp_b_path
args_circuit.client_a = 0    # First client from exp_a
args_circuit.client_b = 0    # First client from exp_b
args_circuit.mode = "circuit"
args_circuit.source = "local"
args_circuit.round = None
args_circuit.round_key = "last"
args_circuit.classes = None  # Compute all overlapping classes automatically
args_circuit.layer = None    # Default: last conv/linear before head
args_circuit.max_samples = 2000
args_circuit.output = "cka_heatmap.png"

cka_exp_circuit = CKACompareExperiment(args_circuit)
cka_exp_circuit.run()

## 6 · View Heatmap

In [ ]:
from IPython.display import Image, display
if os.path.exists("cka_heatmap.png"):
    display(Image(filename="cka_heatmap.png"))
else:
    print("Heatmap not generated. Check if there were any overlapping classes.")